# MiniDance · 训练（VS Code + Colab 运行时）

本地用 VS Code 打开这个文件，代码在 Colab 的 GPU 上运行。

**连接方法**：右上角「选择内核」→ Colab → 新建 Colab 服务器 → 选 **A100**（L4 也能跑，把 `BATCH` 调小）。

注意：代码运行在 Colab 的机器上，**看不到你电脑上的文件**，所以代码从 GitHub 拉取，数据从 Google 网盘读取。

需要网盘里已有：
- `MyDrive/minidance/latents/`：编码好的 latent（86 个分片，约 9GB）
- `MyDrive/minidance/short_captions.jsonl`：短字幕（可选，没有就只用长字幕）

checkpoint 和样例 GIF 都存到网盘，断开后重新运行会自动续训。

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!free -g | head -2

## 1. 挂载网盘，获取代码

In [ ]:
REPO_URL = 'https://github.com/JuniverseLi/minidance.git'
DRIVE_DIR = '/content/drive/MyDrive/minidance'

import os
if not os.path.exists('/content/drive/MyDrive'):
    try:
        from google.colab import drive
        drive.mount('/content/drive')
    except Exception as e:
        raise RuntimeError(f'网盘挂载失败（{type(e).__name__}: {e}）。'
                           '如果 VS Code 里挂载不了，请改用 Colab 网页版运行 notebooks/train_colab.ipynb') from e
assert os.path.exists(f'{DRIVE_DIR}/latents'), f'网盘里没找到 {DRIVE_DIR}/latents'
print('网盘已挂载:', os.listdir(DRIVE_DIR))

if not os.path.exists('/content/minidance'):
    !git clone -q {REPO_URL} /content/minidance
else:
    !git -C /content/minidance fetch -q origin
    !git -C /content/minidance reset -q --hard origin/main
%cd /content/minidance
!git log --oneline -1

In [ ]:
!pip install -q diffusers safetensors "transformers>=4.51"
import transformers; print('transformers', transformers.__version__)

## 2. 把数据从网盘复制到 Colab 本地磁盘

直接从网盘读太慢。约 9GB，复制一次大约几分钟；已经复制过的文件会跳过。

In [ ]:
import glob, time
SRC, DST = f'{DRIVE_DIR}/latents', '/content/latents'
os.makedirs(DST, exist_ok=True)
print(len(glob.glob(f'{SRC}/shard_*.safetensors')), 'shards on Drive')
t = time.time()
!cp -n {SRC}/shard_* {DST}/
n = len(glob.glob(f'{DST}/shard_*.safetensors'))
size = sum(os.path.getsize(p) for p in glob.glob(f'{DST}/shard_*')) / 2**30
print(f'{n} shards, {size:.1f} GB local, copy took {time.time() - t:.0f}s')

CAPS = f'{DRIVE_DIR}/short_captions.jsonl'
if os.path.exists(CAPS):
    print('短字幕:', sum(1 for _ in open(CAPS, encoding='utf-8')), '条')
else:
    print('未找到短字幕文件，只用长字幕训练')
    CAPS = 'none'

## 3. 训练

- `MINUTES`：训练多久后自动停止（会先生成样例、保存 checkpoint）。S4 先跑 60 分钟。
- `BATCH`：A100 40GB 用 64；L4 24GB 用 32；显存不够（CUDA out of memory）就减半。
- 每 1000 步生成一次样例 GIF，存到 `DRIVE_DIR/runs/<RUN>/samples/`。
- `--compile` 能提速，如果报错就把它删掉再运行。
- 中途断开：重新运行全部单元格，会从网盘里的 checkpoint 继续。

In [ ]:
# RUN = 's4_base'
# MINUTES = 60
# BATCH = 32
RUN = 's4_base'    # 不要改名，改了就会从头开始训练
MINUTES = 540      # 这次训练 9 小时
STEPS = 64000      # 学习率衰减的总步数（约 10 小时）
BATCH = 32         # 保持和上次一样
OUT = f'{DRIVE_DIR}/runs/{RUN}'

!python trainer/train_pretrain.py --data_dir /content/latents --short_captions {CAPS}     --preset base --batch {BATCH} --lr 2e-4 --warmup 1000 --steps {STEPS}     --max_minutes {MINUTES} --sample_every 1000 --save_every 2000 --log_every 50     --compile --resume --out {OUT}

## 4. 查看样例

In [ ]:
from IPython.display import Image, display
gifs = sorted(glob.glob(f'{OUT}/samples/*.gif'))
print(len(gifs), 'sample GIFs; 每个 GIF 从左到右：4 条固定提示词 + 2 条训练集字幕')
for g in gifs[-3:]:
    print(os.path.basename(g))
    display(Image(filename=g))

## 5. 训练日志摘要（发给我看这一格的输出就行）

In [ ]:
!grep "DiT-\|step \|reached\|Traceback\|Error" {OUT}/train.log | cut -c1-160 | head -3
print('...')
!grep "step \|reached\|samples ->" {OUT}/train.log | cut -c1-160 | tail -8

## SFT：在高质量片段上精调

预训练完成后再跑这一节（不用重跑上面的训练格，但要先跑完第 1、2 节）。

- 从预训练的最终权重（EMA）出发，只用**高美学分、有运动**的片段（约 1.1 万条，前 30%），只用**短字幕**（中英文），学习率 5e-5。
- 冰川、星空、雾这类少见场景会被多采样一些。
- 3000 步，L4 + batch 32 约 30 分钟；每 500 步出一张样例 GIF、存一次 checkpoint，断了重跑这一格会接着训。
- 结果在 `DRIVE_DIR/runs/sft_base/`，预训练的 `runs/s4_base/` 不会被改动。


In [ ]:
SFT_RUN = 'sft_base'
INIT = f'{DRIVE_DIR}/runs/s4_base/ckpt.pt'   # 预训练的最终权重
SFT_OUT = f'{DRIVE_DIR}/runs/{SFT_RUN}'
assert os.path.exists(INIT), f'没找到预训练权重 {INIT}'
assert CAPS != 'none', 'SFT 需要短字幕文件 short_captions.jsonl'

!python trainer/train_sft.py --data_dir /content/latents --short_captions {CAPS}     --init {INIT} --resume --preset base --batch 32 --steps 3000     --max_minutes 90 --compile --out {SFT_OUT}


In [ ]:
from IPython.display import Image, display
gifs = sorted(glob.glob(f'{SFT_OUT}/samples/*.gif'))
print(len(gifs), 'SFT sample GIFs')
for g in gifs[-2:]:
    print(os.path.basename(g))
    display(Image(filename=g))
